In [1]:
!poetry install -q

In [2]:
"""
환경 설정 및 의존성 주입
- 목적: 프로젝트 경로 인식, 환경 변수 로드, 그리고 로컬 테스트를 위한 Docker DNS 우회
"""
import os
import sys
import datetime
import pandas as pd
from dotenv import load_dotenv

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

env_path = os.path.join(project_root, '.env')
load_dotenv(dotenv_path=env_path)

# [핵심 수정 사항] 
# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

# 2. 모듈 임포트
from src.common.config import ConfigManager
from src.reader.reader_service import ReaderService
from src.transformer.transformer_service import TransformerService

print("✅ 환경 설정 및 모듈 임포트 완료")

✅ 환경 설정 및 모듈 임포트 완료


In [3]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [4]:
"""
5종 API 파티션 데이터 스트리밍 추출
- 목적: KIS, FRED, ECOS, UPBIT의 대표 job_id를 순회하며 파티션 단위 스트리밍을 통해 원본 DataFrame을 획득합니다.
"""

# 1. ReaderService 초기화
reader = ReaderService(target_reader="s3")

# 2. 테스트할 프로바이더 및 job_id 리스트 정의
target_jobs = [
    ("kis", "kis_kospi_finance_daily"),
    ("kis", "kis_nasdaq_daily"),
    ("fred", "fred_us_treasury_10y_daily"),
    ("ecos", "ecos_ktb_10y_daily"),
    ("upbit", "upbit_krw_btc_daily")
]

year = datetime.datetime.now().year
month = datetime.datetime.now().month
day = 13#datetime.datetime.now().day

# 공통 파티션 날짜 (테스트 기준일)
base_date_path = f"year={year}/month={month:02d}/day={day:02d}"
print(f"📅 테스트 기준 날짜 파티션: {base_date_path}")
# 3. 데이터 추출 및 확인
raw_dataframes = {}

for provider, job_id in target_jobs:
    # 파티션 기반 동적 S3 Key(Prefix) 생성
    s3_key = f"raw/provider={provider}/job={job_id}/{base_date_path}"
    
    records = []
    try:
        # Paginator를 통한 스트림 추출
        for batch in reader.read_stream(source_path=s3_key):
            records.extend(batch)
            
        df = pd.DataFrame(records)
        raw_dataframes[job_id] = df  # 다음 셀(변환 테스트)에서 사용하기 위해 딕셔너리에 저장
        
        print(f"\n[{job_id}] 원본 데이터 추출 성공 - 총 레코드 수: {len(df)}")
        display(df.tail(1))

    except Exception as e:
        print(f"❌ 추출 실패: {e}")

📅 테스트 기준 날짜 파티션: year=2026/month=05/day=13
[2026-05-17 00:48:42]  INFO   | ReaderService        | [e7c292fb] 데이터 스트림 추출 요청 위임 - Target: S3, Path: raw/provider=kis/job=kis_kospi_finance_daily/year=2026/month=05/day=13, Batch: 10000
[2026-05-17 00:48:42]  INFO   | ReaderService        | [e7c292fb] [S3] 리더 인스턴스 지연 초기화 진입
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] LocalStack S3 Endpoint로 클라이언트 초기화 (http://localhost:4566)
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 스트리밍 읽기 시작 - Bucket: data-pipeline-bronze, Key: raw/provider=kis/job=kis_kospi_finance_daily/year=2026/month=05/day=13
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 파티션 스트리밍 완료 - 처리된 파일: 7개, 총 레코드: 7건

[kis_kospi_finance_daily] 원본 데이터 추출 성공 - 총 레코드 수: 7


,output1,output2,rt_cd,msg_cd,msg1
6,"{'bstp_nmix_prdy_vrss': '5.58', 'prdy_vrss_sign': '2', 'bstp_nmix_prdy_ctrt': '0.47', 'prdy_nmix': '1183.34', 'acml_vol': '65541', 'acml_tr_pbmn': '3619507', 'hts_kor_isnm': '금융', 'bstp_nmix_prpr': '1188.92', 'bstp_cls_code': '0021', 'prdy_vol': '112590', 'bstp_nmix_oprc': '1178.43', 'bstp_nmix_hgpr': '1190.28', 'bstp_nmix_lwpr': '1154.78', 'futs_prdy_oprc': '1236.46', 'futs_prdy_hgpr': '1242.67', 'futs_prdy_lwpr': '1156.41'}","[{'stck_bsop_date': '20260513', 'bstp_nmix_prpr': '1188.92', 'bstp_nmix_oprc': '1178.43', 'bstp_nmix_hgpr': '1190.28', 'bstp_nmix_lwpr': '1154.78', 'acml_vol': '65541', 'acml_tr_pbmn': '3619507', 'mod_yn': 'N'}, {'stck_bsop_date': '20260512', 'bstp_nmix_prpr': '1183.34', 'bstp_nmix_oprc': '1236.46', 'bstp_nmix_hgpr': '1242.67', 'bstp_nmix_lwpr': '1156.41', 'acml_vol': '112590', 'acml_tr_pbmn': '5473067', 'mod_yn': 'N'}]",0,MCA00000,정상처리 되었습니다.


[2026-05-17 00:48:42]  INFO   | ReaderService        | [e7c292fb] 데이터 스트림 추출 요청 위임 - Target: S3, Path: raw/provider=kis/job=kis_nasdaq_daily/year=2026/month=05/day=13, Batch: 10000
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 스트리밍 읽기 시작 - Bucket: data-pipeline-bronze, Key: raw/provider=kis/job=kis_nasdaq_daily/year=2026/month=05/day=13
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 파티션 스트리밍 완료 - 처리된 파일: 8개, 총 레코드: 8건

[kis_nasdaq_daily] 원본 데이터 추출 성공 - 총 레코드 수: 8


,output1,output2,rt_cd,msg_cd,msg1
7,"{'ovrs_nmix_prdy_vrss': '-410.07', 'prdy_vrss_sign': '5', 'prdy_ctrt': '-1.54', 'ovrs_nmix_prdy_clpr': '26635.22', 'acml_vol': '9904915300', 'hts_kor_isnm': '나스닥 종합', 'ovrs_nmix_prpr': '26225.15', 'stck_shrn_iscd': 'COMP', 'ovrs_prod_oprc': '26288.92', 'ovrs_prod_hgpr': '26460.76', 'ovrs_prod_lwpr': '26097.54'}","[{'stck_bsop_date': '20260513', 'ovrs_nmix_prpr': '26402.34', 'ovrs_nmix_oprc': '26147.65', 'ovrs_nmix_hgpr': '26474.18', 'ovrs_nmix_lwpr': '25990.16', 'acml_vol': '10131419300', 'mod_yn': 'N'}, {'stck_bsop_date': '20260512', 'ovrs_nmix_prpr': '26088.20', 'ovrs_nmix_oprc': '26087.01', 'ovrs_nmix_hgpr': '26190.48', 'ovrs_nmix_lwpr': '25739.22', 'acml_vol': '9999547200', 'mod_yn': 'N'}]",0,MCA00000,정상처리 되었습니다.


[2026-05-17 00:48:42]  INFO   | ReaderService        | [e7c292fb] 데이터 스트림 추출 요청 위임 - Target: S3, Path: raw/provider=fred/job=fred_us_treasury_10y_daily/year=2026/month=05/day=13, Batch: 10000
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 스트리밍 읽기 시작 - Bucket: data-pipeline-bronze, Key: raw/provider=fred/job=fred_us_treasury_10y_daily/year=2026/month=05/day=13
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 파티션 스트리밍 완료 - 처리된 파일: 6개, 총 레코드: 6건

[fred_us_treasury_10y_daily] 원본 데이터 추출 성공 - 총 레코드 수: 6


,realtime_start,realtime_end,observation_start,observation_end,units,output_type,file_type,order_by,sort_order,count,offset,limit,observations
5,2026-05-16,2026-05-16,2026-05-12,2026-05-13,lin,1,json,observation_date,asc,2,0,100000,"[{'realtime_start': '2026-05-16', 'realtime_end': '2026-05-16', 'date': '2026-05-12', 'value': '4.46'}, {'realtime_start': '2026-05-16', 'realtime_end': '2026-05-16', 'date': '2026-05-13', 'value': '4.46'}]"


[2026-05-17 00:48:42]  INFO   | ReaderService        | [e7c292fb] 데이터 스트림 추출 요청 위임 - Target: S3, Path: raw/provider=ecos/job=ecos_ktb_10y_daily/year=2026/month=05/day=13, Batch: 10000
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 스트리밍 읽기 시작 - Bucket: data-pipeline-bronze, Key: raw/provider=ecos/job=ecos_ktb_10y_daily/year=2026/month=05/day=13
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 파티션 스트리밍 완료 - 처리된 파일: 8개, 총 레코드: 8건

[ecos_ktb_10y_daily] 원본 데이터 추출 성공 - 총 레코드 수: 8


,StatisticSearch
7,"{'list_total_count': 2, 'row': [{'STAT_CODE': '817Y002', 'STAT_NAME': '1.3.2.1. 시장금리(일별)', 'ITEM_CODE1': '010210000', 'ITEM_NAME1': '국고채(10년)', 'ITEM_CODE2': None, 'ITEM_NAME2': None, 'ITEM_CODE3': None, 'ITEM_NAME3': None, 'ITEM_CODE4': None, 'ITEM_NAME4': None, 'UNIT_NAME': '연%', 'WGT': None, 'TIME': '20260512', 'DATA_VALUE': '4.056'}, {'STAT_CODE': '817Y002', 'STAT_NAME': '1.3.2.1. 시장금리(일별)', 'ITEM_CODE1': '010210000', 'ITEM_NAME1': '국고채(10년)', 'ITEM_CODE2': None, 'ITEM_NAME2': None, 'ITEM_CODE3': None, 'ITEM_NAME3': None, 'ITEM_CODE4': None, 'ITEM_NAME4': None, 'UNIT_NAME': '연%', 'WGT': None, 'TIME': '20260513', 'DATA_VALUE': '4.044'}]}"


[2026-05-17 00:48:42]  INFO   | ReaderService        | [e7c292fb] 데이터 스트림 추출 요청 위임 - Target: S3, Path: raw/provider=upbit/job=upbit_krw_btc_daily/year=2026/month=05/day=13, Batch: 10000
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 스트리밍 읽기 시작 - Bucket: data-pipeline-bronze, Key: raw/provider=upbit/job=upbit_krw_btc_daily/year=2026/month=05/day=13
[2026-05-17 00:48:42]  INFO   | S3ZstdStreamingReader | [e7c292fb] [S3_BRONZE_READER] S3 파티션 스트리밍 완료 - 처리된 파일: 8개, 총 레코드: 8건

[upbit_krw_btc_daily] 원본 데이터 추출 성공 - 총 레코드 수: 8


,0
7,"{'market': 'KRW-BTC', 'candle_date_time_utc': '2026-05-16T00:00:00', 'candle_date_time_kst': '2026-05-16T09:00:00', 'opening_price': 117783000.0, 'high_price': 118050000.0, 'low_price': 115833000.0, 'trade_price': 116179000.0, 'timestamp': 1778938366403, 'candle_acc_trade_price': 101582434255.12816, 'candle_acc_trade_volume': 869.6921828, 'prev_closing_price': 117780000.0, 'change_price': -1601000.0, 'change_rate': -0.0135931398}"


In [5]:
"""
[Cell 3] 5종 API 통합 변환(Transformer) 테스트
- 목적: Cell 2에서 추출한 원본 데이터를 스트리밍으로 전달하여 Silver 데이터로 변환합니다.
- 특징: S3 재호출 없이 메모리 내의 DataFrame을 제너레이터로 래핑(iter)하여 파이프라인을 고속 모사합니다.
"""
from src.transformer.transformer_service import TransformerService
import pandas as pd
from IPython.display import display

# 1. Transformer 서비스 초기화
transformer_service = TransformerService()
ENFORCE_SCHEMA = True  # Production 모드 (Silver 스키마 강제 적용)

for job_id, raw_df in raw_dataframes.items():
    print(f"\n" + "━" * 80)
    print(f"🚀 [변환 파이프라인] {job_id}")
    print("━" * 80)
    
    try:
        # 2. DataFrame을 단일 청크 스트림(Iterator)으로 래핑하여 메모리 낭비 없이 제너레이터 주입
        raw_stream = iter([raw_df])
        
        # 3. 스트리밍 변환 실행
        transformed_stream = transformer_service.transform_stream(
            job_id=job_id, 
            data_stream=raw_stream,
            enforce_schema=ENFORCE_SCHEMA
        )
        
        # 4. 결과 병합 (리스트 컴프리헨션 대체로 코드 간결화)
        transformed_chunks = list(transformed_stream)
        
        if transformed_chunks:
            silver_df = pd.concat(transformed_chunks, ignore_index=True)
            print(f"✅ 변환 성공 | 크기: {len(silver_df)} 행 × {len(silver_df.columns)} 열")
            print(f"📌 데이터 타입 스키마:\n{silver_df.dtypes.to_string()}\n")
            display(silver_df.tail(1))
        else:
            print("⚠️ 변환된 데이터가 없습니다 (빈 데이터프레임).")
            
    except Exception as e:
        # 아직 변환기가 구현되지 않은 API(FRED, ECOS 등)는 여기서 명확히 에러를 뱉어냅니다.
        print(f"❌ 변환 실패 (Fail-Fast 정상 작동): {e}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] kis_kospi_finance_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [kis_kospi_finance_daily] 변환기 지연 초기화 진입 (Policy: kis_domestic_schema)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [kis_kospi_finance_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [kis_kospi_finance_daily] 스트리밍 변환 완료 (총 1개 청크 처리됨)
✅ 변환 성공 | 크기: 9 행 × 14 열
📌 데이터 타입 스키마:
trade_date           datetime64[us]
price_change_sign            string
close                       float32
open                        float32
high                        float32
low                         float32
prev_price                  float32
price_change_rate           float32
volume                      float32
prev_volume                 float32
trading_value

,trade_date,price_change_sign,close,open,high,low,prev_price,price_change_rate,volume,prev_volume,trading_value,futures_prev_open,futures_prev_high,futures_prev_low
8,2026-05-12,2,1183.339966,1236.459961,1242.670044,1156.410034,1183.339966,0.47,112590.0,112590.0,5473067.0,1236.459961,1242.670044,1156.410034



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] kis_nasdaq_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [kis_nasdaq_daily] 변환기 지연 초기화 진입 (Policy: kis_overseas_schema)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [kis_nasdaq_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [kis_nasdaq_daily] 스트리밍 변환 완료 (총 1개 청크 처리됨)
✅ 변환 성공 | 크기: 8 행 × 10 열
📌 데이터 타입 스키마:
trade_date           datetime64[us]
price_change_sign            string
close                       float32
open                        float32
high                        float32
low                         float32
prev_price                  float32
price_change                float32
price_change_rate           float32
volume                      float32



,trade_date,price_change_sign,close,open,high,low,prev_price,price_change,price_change_rate,volume
7,2026-05-12,5,26088.199219,26087.009766,26190.480469,25739.220703,26635.220703,-410.070007,-1.54,9.999547e+09



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] fred_us_treasury_10y_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [fred_us_treasury_10y_daily] 변환기 지연 초기화 진입 (Policy: fred_base_schema)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [fred_us_treasury_10y_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [fred_us_treasury_10y_daily] 스트리밍 변환 완료 (총 1개 청크 처리됨)
✅ 변환 성공 | 크기: 3 행 × 2 열
📌 데이터 타입 스키마:
trade_date    datetime64[us]
value                float32



,trade_date,value
2,2026-05-13,4.46



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] ecos_ktb_10y_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [ecos_ktb_10y_daily] 변환기 지연 초기화 진입 (Policy: ecos_base_schema)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [ecos_ktb_10y_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [ecos_ktb_10y_daily] 스트리밍 변환 완료 (총 1개 청크 처리됨)
✅ 변환 성공 | 크기: 14 행 × 2 열
📌 데이터 타입 스키마:
trade_date    datetime64[us]
value                float32



,trade_date,value
13,2026-05-13,4.044



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] upbit_krw_btc_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [upbit_krw_btc_daily] 변환기 지연 초기화 진입 (Policy: upbit_base_schema)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [upbit_krw_btc_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-17 00:48:42]  INFO   | TransformerService   | [e7c292fb] [upbit_krw_btc_daily] 스트리밍 변환 완료 (총 1개 청크 처리됨)
✅ 변환 성공 | 크기: 8 행 × 10 열
📌 데이터 타입 스키마:
trade_date       datetime64[us]
open                    float32
high                    float32
low                     float32
close                   float32
prev_close              float32
change_price            float32
change_rate             float32
trading_value           float64
volume                  float64



,trade_date,open,high,low,close,prev_close,change_price,change_rate,trading_value,volume
7,2026-05-16 09:00:00,117783000.0,118050000.0,115833000.0,116179000.0,117780000.0,-1601000.0,-0.013593,1.015824e+11,869.692183
